[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bluerrror/germany-hydrology/blob/main/examples/basin_explorer.ipynb)

# 🗺️ Basin explorer — pick a basin, get its data, calibrate HBV

Everything in this notebook is interactive (run it with a live kernel):

1. choose a **basin level** (the map redraws) and **click a catchment** —
   or pick a gauge and its basin auto-selects;
2. choose a source per domain and press **Fetch basin data**;
3. explore the tabs: climate time series with a **variable dropdown**,
   land-cover **bar chart**, soil composition, **hydrograph**, and a
   zoomable **hillshaded DEM** — all plotly;
4. in the **HBV** tab: pick the **training range with a slider**, the
   number of **Optuna trials** and the objective, press *Calibrate* — the
   hydrograph shows observed vs simulated with **NSE / KGE boxes** for the
   training and test periods.

In [1]:
# Setup — no-op locally; on Colab installs the plugins + widgets + plotly
import importlib.util, sys

if importlib.util.find_spec("germany_hydrology") is None:
    %pip install -q git+https://github.com/Bluerrror/germany-hydrology ipyleaflet ipywidgets plotly optuna
if importlib.util.find_spec("earthkit_data_soilgrids") is None:
    %pip install -q git+https://github.com/Bluerrror/earthkit-data-soilgrids
if "google.colab" in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import json

import earthkit.data as ekd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ekd.config.set("cache-policy", "user")

from germany_hydrology import GERMANY, signatures as hs
from germany_hydrology.landcover import CLASSES

# official ESA WorldCover class colors (identity colors of the product)
WC_COLORS = {10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff",
             50: "#fa0000", 60: "#b4b4b4", 70: "#f0f0f0", 80: "#0064c8",
             90: "#0096a0", 95: "#00cf75", 100: "#fae6a0"}

basins = ekd.from_source("hydrosheds", product="basins", level=7, bbox=GERMANY).to_pandas()
gauges = ekd.from_source("pegelonline").to_pandas().dropna(subset=["longitude", "latitude"])
print(f"{len(basins)} basins, {len(gauges)} gauges loaded")

252 basins, 729 gauges loaded


## Fetch helpers — plain functions, usable in your own scripts too

In [3]:
def fetch_climate(source, basin, start="2015-01-01", end="2023-12-31", hyras_year=2020):
    lon, lat = basin.geometry.centroid.x, basin.geometry.centroid.y
    if source == "ERA5 (centroid)":
        return ekd.from_source(
            "era5-timeseries", latitude=lat, longitude=lon, start=start, end=end,
            variables=["precipitation_sum", "temperature_2m_mean",
                       "temperature_2m_max", "temperature_2m_min",
                       "et0_fao_evapotranspiration", "snowfall_sum"],
        ).to_pandas()
    if source == "DWD station (nearest)":
        st = ekd.from_source("dwd-observations").to_pandas()
        st = st[st["to_date"] > "2020"]
        sid = ((st["latitude"] - lat) ** 2 + (st["longitude"] - lon) ** 2).idxmin()
        df = ekd.from_source("dwd-observations", station=sid, period="all").to_pandas()
        df.attrs["station"] = f"{st.loc[sid, 'name']} ({sid})"
        return df.select_dtypes("number")
    if source == "HYRAS grid (1 km)":
        w, s, e, n = basin.geometry.bounds
        return ekd.from_source("dwd-grids", variable="precipitation",
                               years=hyras_year, bbox=[w, s, e, n]).to_xarray()
    raise ValueError(source)


def fetch_landcover(source, basin):
    year = int(source.split()[-1])
    w, s, e, n = basin.geometry.bounds
    da = ekd.from_source("worldcover", bbox=[w, s, e, n], year=year).to_xarray()
    da = da.rio.clip([basin.geometry], crs="EPSG:4326", drop=True)
    vals, counts = np.unique(da.values, return_counts=True)
    frac = pd.Series(counts, index=vals).drop(0, errors="ignore")  # 0 = nodata
    frac = (frac / frac.sum() * 100).round(2)
    colors = [WC_COLORS.get(v, "#999999") for v in frac.index]
    frac.index = [CLASSES.get(v, str(v)) for v in frac.index]
    return da, frac, colors


def fetch_soil(source, basin):
    w, s, e, n = basin.geometry.bounds
    if source == "BÜK1000 (polygons)":
        gdf = ekd.from_source("buek1000", bbox=[w, s, e, n]).to_pandas()
        gdf = gdf.clip(basin.geometry)
        area = gdf.to_crs("EPSG:3035").area
        frac = (area.groupby(gdf["Symbol"]).sum() / area.sum() * 100)
        return gdf, frac.sort_values(ascending=False).round(2)
    prop = {"SoilGrids clay": "clay", "SoilGrids sand": "sand",
            "SoilGrids pH": "phh2o", "SoilGrids SOC": "soc"}[source]
    da = ekd.from_source("soilgrids", property=prop, depth="0-5cm",
                         stat="mean", bbox=[w, s, e, n]).to_xarray()["band_1"]
    return (da.where(da != 0) / 10.0).rename(prop), None


def fetch_hydrology(source, gauge_name):
    if gauge_name is None:
        raise ValueError("No gauge inside/near this basin — pick one on the map.")
    parameter = "W" if "level" in source else "Q"
    df = ekd.from_source("pegelonline", station=gauge_name,
                         parameter=parameter, start="P30D").to_pandas()
    df.index = df.index.tz_convert("UTC").tz_localize(None)
    return df


def fetch_terrain(source, basin):
    res = 90 if "90" in source else 30
    w, s, e, n = basin.geometry.bounds
    da = ekd.from_source("copernicus-dem", bbox=[w, s, e, n], resolution=res).to_xarray()
    return da.rio.clip([basin.geometry], crs="EPSG:4326", drop=True)


def fetch_hbv_data(basin, gauge_name):
    """Discharge (mm/day, via the basin's upstream area) + ERA5 forcing."""
    q = fetch_hydrology("Discharge (Q)", gauge_name)
    q_daily = q.iloc[:, 0].resample("D").mean().dropna()
    area_km2 = float(basin.get("UP_AREA") or basin["SUB_AREA"])
    q_mm = q_daily * 86.4 / area_km2

    lon, lat = basin.geometry.centroid.x, basin.geometry.centroid.y
    end = (pd.Timestamp.now() - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    met = ekd.from_source(
        "era5-timeseries", latitude=lat, longitude=lon,
        start="2018-01-01", end=end,
        variables=["precipitation_sum", "temperature_2m_mean",
                   "et0_fao_evapotranspiration"],
    ).to_pandas()
    return met, q_mm.reindex(met.index)

## Interactive renderers (plotly)

In [4]:
from matplotlib import colormaps
from matplotlib.colors import LightSource

PLOT_H = 340


def climate_figure(result, variable, freq="MS"):
    if hasattr(result, "data_vars"):                        # HYRAS Dataset
        series = result[variable].mean(dim=("x", "y")).to_series()
        title = f"HYRAS basin-mean {variable}"
    else:
        agg = "sum" if ("precip" in variable or "snow" in variable
                        or variable in ("RSK", "RS")) else "mean"
        series = getattr(result[variable].resample(freq), agg)()
        title = f"{variable} — monthly {agg}"
    fig = px.line(x=series.index, y=series.values,
                  labels={"x": "", "y": variable}, title=title)
    fig.update_traces(line=dict(color="#31688e", width=1.6))
    fig.update_layout(height=PLOT_H, margin=dict(t=45, b=10))
    return fig


def landcover_figure(frac, colors):
    fig = go.Figure(go.Bar(x=frac.values, y=frac.index, orientation="h",
                           marker_color=colors))
    fig.update_layout(title="Land cover composition (ESA WorldCover)",
                      xaxis_title="% of basin area", height=PLOT_H,
                      margin=dict(t=45, b=10),
                      yaxis=dict(autorange="reversed"))
    return fig


def soil_figure(result, frac, source_label):
    if frac is not None:                                    # BÜK1000
        top = frac.head(10)
        fig = go.Figure(go.Bar(x=top.values, y=[str(i) for i in top.index],
                               orientation="h", marker_color="#8c6d31"))
        fig.update_layout(title="Soil units (BÜK1000, % of basin, top 10)",
                          xaxis_title="% of basin area", height=PLOT_H,
                          margin=dict(t=45, b=10),
                          yaxis=dict(autorange="reversed"))
        return fig
    da = result.coarsen(x=max(1, result.sizes["x"] // 400),
                        y=max(1, result.sizes["y"] // 400), boundary="trim").mean()
    fig = go.Figure(go.Heatmap(z=da.values, x=da.x.values, y=da.y.values,
                               colorscale="Viridis", colorbar_title=source_label))
    fig.update_layout(title=f"{source_label} (0–5 cm)", height=PLOT_H + 60,
                      yaxis=dict(scaleanchor="x"), margin=dict(t=45, b=10))
    return fig


def hydrology_figure(df, gauge):
    fig = px.line(x=df.index, y=df.iloc[:, 0].values,
                  labels={"x": "", "y": df.columns[0]},
                  title=f"{gauge} — last 30 days ({df.columns[0]})")
    fig.update_traces(line=dict(color="#31688e", width=1.3))
    fig.update_layout(height=PLOT_H, margin=dict(t=45, b=10))
    return fig


def terrain_figure(dem):
    z = dem.values.astype(float)
    ls = LightSource(azdeg=315, altdeg=45)
    rgb = ls.shade(np.where(np.isfinite(z), z, np.nanmin(z)),
                   cmap=colormaps["terrain"], blend_mode="overlay",
                   vert_exag=8)
    rgb[..., 3] = np.where(np.isfinite(z), 1.0, 0.0)
    fig = px.imshow(rgb, x=dem.x.values, y=dem.y.values, origin="upper",
                    title="Elevation, hillshaded (Copernicus DEM)")
    fig.update_layout(height=PLOT_H + 140, margin=dict(t=45, b=10),
                      xaxis_title="Longitude", yaxis_title="Latitude")
    return fig


def hbv_figure(obs, sim, train_range, metrics):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=obs.index, y=obs.values, name="observed",
                             line=dict(color="#444444", width=2)))
    fig.add_trace(go.Scatter(x=sim.index, y=sim.values, name="HBV simulated",
                             line=dict(color="#31688e", width=2)))
    t0, t1 = train_range
    fig.add_vrect(x0=t0, x1=t1, fillcolor="#31688e", opacity=0.07,
                  line_width=0)
    fig.add_vrect(x0=t1, x1=obs.index.max(), fillcolor="#d95f02", opacity=0.07,
                  line_width=0)
    for label, (mid, m) in metrics.items():
        fig.add_annotation(x=mid, y=1.13, yref="paper", showarrow=False,
                           text=(f"<b>{label}</b><br>NSE {m['nse']:.2f} · "
                                 f"KGE {m['kge']:.2f}"),
                           bgcolor="white", bordercolor="#888888",
                           borderwidth=1, font=dict(size=11))
    fig.update_layout(title="HBV — observed vs simulated (mm/day)",
                      height=PLOT_H + 60, margin=dict(t=80, b=10),
                      yaxis_title="Q (mm/day)",
                      legend=dict(orientation="h", y=-0.15))
    return fig

## The explorer

In [5]:
import ipywidgets as W
from ipyleaflet import GeoJSON, Map, basemaps
from shapely.geometry import Point

m = Map(center=(51.2, 10.3), zoom=6, basemap=basemaps.OpenStreetMap.Mapnik,
        scroll_wheel_zoom=True, layout=W.Layout(height="440px"))

display_basins = basins[["HYBAS_ID", "geometry"]].copy()
display_basins["geometry"] = display_basins.geometry.simplify(0.01)
basin_layer = GeoJSON(data=json.loads(display_basins.to_json()),
                      style={"color": "#31688e", "weight": 1, "fillOpacity": 0.05},
                      hover_style={"fillOpacity": 0.25})
m.add(basin_layer)
highlight = GeoJSON(data={"type": "FeatureCollection", "features": []},
                    style={"color": "#d95f02", "weight": 3, "fillOpacity": 0.15})
m.add(highlight)

state = {"basin": None, "gauge": None}
basin_data = {}

# ---- controls ---------------------------------------------------------
level_dd = W.Dropdown(options=[4, 5, 6, 7, 8], value=7, description="Basin level")
gauge_dd = W.Dropdown(options=sorted(gauges.index), description="Gauge")
climate_dd = W.Dropdown(options=["ERA5 (centroid)", "DWD station (nearest)",
                                 "HYRAS grid (1 km)"], description="Climate")
lc_dd = W.Dropdown(options=["WorldCover 2021", "WorldCover 2020"], description="Land cover")
soil_dd = W.Dropdown(options=["BÜK1000 (polygons)", "SoilGrids clay", "SoilGrids sand",
                              "SoilGrids pH", "SoilGrids SOC"], description="Soil")
hydro_dd = W.Dropdown(options=["Water level (W)", "Discharge (Q)"], description="Hydrology")
terrain_dd = W.Dropdown(options=["DEM 90 m", "DEM 30 m"], description="Terrain")
domains = {name: W.Checkbox(value=name in ("climate", "landcover"), description=name,
                            indent=False, layout=W.Layout(width="105px"))
           for name in ("climate", "landcover", "soil", "hydrology", "terrain")}
fetch_btn = W.Button(description="Fetch basin data", button_style="primary", icon="download")
status = W.HTML("<i>Click a basin on the map, or pick a gauge.</i>")

# ---- per-tab outputs (climate gets its own variable dropdown) ---------
out = {k: W.Output() for k in ("climate", "landcover", "soil", "hydrology",
                               "terrain", "hbv")}
climate_var_dd = W.Dropdown(description="Variable", layout=W.Layout(width="320px"))

# HBV controls
trials_slider = W.IntSlider(value=60, min=10, max=300, step=10,
                            description="Optuna trials", continuous_update=False)
objective_dd = W.Dropdown(options=["rmse", "nse", "kge", "log_nse"], value="rmse",
                          description="Objective")
train_slider = W.SelectionRangeSlider(options=[("—", 0), ("—?", 1)], index=(0, 1),
                                      description="Train range",
                                      continuous_update=False,
                                      layout=W.Layout(width="620px"))
prep_btn = W.Button(description="1) Load gauge + forcing", icon="download")
calib_btn = W.Button(description="2) Calibrate HBV", button_style="warning",
                     icon="cogs", disabled=True)
hbv_status = W.HTML("")

tabs = W.Tab(children=[
    W.VBox([climate_var_dd, out["climate"]]),
    out["landcover"], out["soil"], out["hydrology"], out["terrain"],
    W.VBox([W.HBox([prep_btn, trials_slider, objective_dd, calib_btn]),
            train_slider, hbv_status, out["hbv"]]),
])
for i, t in enumerate(["climate", "landcover", "soil", "hydrology", "terrain", "HBV"]):
    tabs.set_title(i, t)

In [6]:
# ---- selection logic --------------------------------------------------

def _select_basin(hybas_id):
    row = basins[basins["HYBAS_ID"] == hybas_id].iloc[0]
    state["basin"] = row
    highlight.data = json.loads(
        display_basins[display_basins["HYBAS_ID"] == hybas_id].to_json())
    w, s, e, n = row.geometry.bounds
    inside = gauges[gauges["longitude"].between(w, e) & gauges["latitude"].between(s, n)]
    gauge_dd.options = sorted(inside.index) or sorted(gauges.index)
    state["gauge"] = gauge_dd.options[0] if len(inside) else None
    status.value = f"Basin <b>{hybas_id}</b> selected ({len(inside)} gauges inside)."


def on_basin_click(event=None, feature=None, **kwargs):
    if feature:
        _select_basin(feature["properties"]["HYBAS_ID"])

basin_layer.on_click(on_basin_click)


def on_gauge_change(change):
    if change["new"] is None:
        return
    state["gauge"] = change["new"]
    g = gauges.loc[change["new"]]
    hit = basins[basins.contains(Point(g["longitude"], g["latitude"]))]
    if len(hit):
        _select_basin(hit["HYBAS_ID"].iloc[0])
    m.center = (float(g["latitude"]), float(g["longitude"]))
    m.zoom = 9

gauge_dd.observe(on_gauge_change, names="value")


def on_level_change(change):
    global basins, display_basins
    lvl = change["new"]
    status.value = f"Loading HydroBASINS level {lvl} …"
    basins = ekd.from_source("hydrosheds", product="basins", level=lvl,
                             bbox=GERMANY).to_pandas()
    display_basins = basins[["HYBAS_ID", "geometry"]].copy()
    display_basins["geometry"] = display_basins.geometry.simplify(0.01)
    basin_layer.data = json.loads(display_basins.to_json())
    highlight.data = {"type": "FeatureCollection", "features": []}
    state["basin"] = None
    status.value = f"Level {lvl}: {len(basins)} basins — click one to select."

level_dd.observe(on_level_change, names="value")


# ---- fetch + render ---------------------------------------------------

def render_climate(change=None):
    if "climate" not in basin_data or climate_var_dd.value is None:
        return
    out["climate"].clear_output()
    with out["climate"]:
        display(climate_figure(basin_data["climate"], climate_var_dd.value))

climate_var_dd.observe(render_climate, names="value")


def on_fetch(_):
    if state["basin"] is None:
        status.value = "<b style='color:#b00'>Select a basin first.</b>"
        return
    b = state["basin"]
    for name, fetcher, render in [
        ("climate", lambda: fetch_climate(climate_dd.value, b), None),
        ("landcover", lambda: fetch_landcover(lc_dd.value, b),
         lambda r: landcover_figure(r[1], r[2])),
        ("soil", lambda: fetch_soil(soil_dd.value, b),
         lambda r: soil_figure(r[0], r[1], soil_dd.value)),
        ("hydrology", lambda: fetch_hydrology(hydro_dd.value, state["gauge"]),
         lambda r: hydrology_figure(r, state["gauge"])),
        ("terrain", lambda: fetch_terrain(terrain_dd.value, b),
         lambda r: terrain_figure(r)),
    ]:
        if not domains[name].value:
            continue
        status.value = f"Fetching <b>{name}</b> …"
        try:
            basin_data[name] = fetcher()
            if name == "climate":
                result = basin_data["climate"]
                opts = (list(result.data_vars) if hasattr(result, "data_vars")
                        else list(result.columns))
                opts = [o for o in opts if o not in ("x_bnds", "y_bnds",
                                                     "time_bnds", "crs")]
                climate_var_dd.options = opts
                climate_var_dd.value = opts[0]
                render_climate()
            else:
                out[name].clear_output()
                with out[name]:
                    display(render(basin_data[name]))
        except Exception as exc:
            out[name].clear_output()
            with out[name]:
                print(f"{name} failed: {exc}")
    basin_data["basin"] = b
    status.value = f"Done — basin_data has: <b>{sorted(basin_data)}</b>"

fetch_btn.on_click(on_fetch)


# ---- HBV: prepare, choose training range, calibrate --------------------

def on_prep(_):
    if state["basin"] is None or state["gauge"] is None:
        hbv_status.value = "<b style='color:#b00'>Select a basin/gauge first.</b>"
        return
    hbv_status.value = f"Loading discharge at <b>{state['gauge']}</b> + ERA5 forcing …"
    try:
        met, obs = fetch_hbv_data(state["basin"], state["gauge"])
    except Exception as exc:
        hbv_status.value = f"<b style='color:#b00'>Failed:</b> {exc} (gauge has no Q?)"
        return
    basin_data["hbv_forcing"], basin_data["hbv_obs"] = met, obs
    days = obs.dropna().index
    train_slider.options = [(d.strftime("%d %b"), d) for d in days]
    train_slider.index = (0, max(1, int(len(days) * 0.6)))
    calib_btn.disabled = False
    hbv_status.value = (f"{len(days)} days of discharge available "
                        f"({days.min().date()} → {days.max().date()}). "
                        "Set the training range, then calibrate.")

prep_btn.on_click(on_prep)


def on_calibrate(_):
    from germany_hydrology import hbv

    met, obs = basin_data["hbv_forcing"], basin_data["hbv_obs"]
    t0, t1 = train_slider.value
    objective = ({"rmse": lambda o, s: -float(np.sqrt(np.mean((o - s) ** 2)))}
                 .get(objective_dd.value, objective_dd.value))
    hbv_status.value = (f"Calibrating ({trials_slider.value} trials, "
                        f"{objective_dd.value}, train {t0.date()} → {t1.date()}) …")
    try:
        result = hbv.calibrate(
            met["precipitation_sum"], met["temperature_2m_mean"],
            met["et0_fao_evapotranspiration"], obs,
            objective=objective, n_trials=trials_slider.value, warmup=365,
            calibration_period=(str(t0.date()), str(t1.date())),
            fixed={"CFR": 0.05, "CWH": 0.1},
        )
    except Exception as exc:
        hbv_status.value = f"<b style='color:#b00'>Calibration failed:</b> {exc}"
        return
    basin_data["hbv"] = result

    days = obs.dropna()
    sim = result["simulation"].reindex(days.index)
    train = days.index <= t1
    metrics = {}
    for label, mask in [("train", train), ("test", ~train)]:
        o, s = days[mask], sim[mask]
        if len(o) >= 3 and o.std() > 0:
            metrics[label] = (o.index[len(o) // 2],
                              {"nse": hs.nse(o, s), "kge": hs.kge(o, s)})
    out["hbv"].clear_output()
    with out["hbv"]:
        display(hbv_figure(days, sim, (t0, t1), metrics))
    hbv_status.value = ("Done. NaN/missing boxes mean the period was too "
                        "short or flow too constant for that metric.")

calib_btn.on_click(on_calibrate)

ui = W.VBox([
    W.HBox([level_dd, gauge_dd]),
    W.HBox([climate_dd, lc_dd, soil_dd]),
    W.HBox([hydro_dd, terrain_dd]),
    W.HBox(list(domains.values()) + [fetch_btn]),
    status, m, tabs,
])
ui

(interactive widget — run the notebook to use it)

Everything fetched lives in `basin_data` (plain pandas / xarray /
GeoDataFrame objects), and the HBV results in `basin_data["hbv"]`
(`best_params`, the optuna `study`, the full `simulation` series).

> The HBV panel calibrates on PEGELONLINE's ~30-day live record — a
> workflow demo. For a proper multi-decade calibration with holdout
> validation, benchmarks and uncertainty bands, see
> [`hbv_calibration.ipynb`](hbv_calibration.ipynb).

Data: DWD (CC-BY 4.0) · WSV PEGELONLINE · HydroSHEDS · BGR BÜK1000 ·
ISRIC SoilGrids (CC-BY 4.0) · ESA WorldCover (CC-BY 4.0) · Copernicus
DEM © DLR/Airbus/ESA · ERA5 via Open-Meteo (CC-BY 4.0).